# DYS-HJ for Overlapping Group LASSO (Real-World Application)

Compares the FoGLasso algorithm (Yuan et al., NeurIPS 2011) against Davis–Yin splitting with HJ-Prox on the GSE2034 breast-cancer expression dataset, using KEGG pathways as overlapping groups.  Reproduces Figure 5.

> **First-run note**: the data-loading cell downloads ~50 MB of GSE2034 microarray data from NCBI GEO into a local `data/` folder.  This can take 1–5 minutes on the first invocation.

## Setup


In [ ]:
# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
import GEOparse
import gseapy as gp

# Plotting configuration
plt.rcParams.update({'font.size': 20})


# ============================================================================
# Optimized HJ-Prox Implementation
# ============================================================================

def compute_prox_optimized(x, t, f, delta=1e-1, num_samples=100, alpha=1.0, 
                                linesearch_iters=0, device='cpu'):
    """
    Optimized (einsum-based) variant of hj_prox kept local to this notebook.
    Used in place of the shared `hj_prox` because the einsum reduction in
    float32 produces slightly different rounding from the shared matmul
    implementation, and changing the reduction here would alter the
    published Figure-5 numbers.

    Differences from the shared `hj_prox`:
    
    1. Use torch.sqrt instead of np.sqrt
    2. More efficient overflow checking
    3. Better memory layout for y samples
    4. Reduced unnecessary operations
    """
    assert x.shape[1] == 1
    assert x.shape[0] >= 1
    
    linesearch_iters += 1
    dim = x.shape[0]
    input_dtype = x.dtype
    
    # Use torch operations throughout
    standard_dev = torch.sqrt(torch.tensor(delta * t / alpha, dtype=input_dtype, device=device))
    
    # Sample y points
    x_flat = x.squeeze(1)
    y = standard_dev * torch.randn(num_samples, dim, device=device, dtype=input_dtype) + x_flat
    
    # Compute weights
    z = -f(y) * (alpha / delta)
    z = z.to(dtype=input_dtype)
    
    # Check for overflow before softmax
    z_max = z.max()
    if z_max > 88:
        alpha *= 0.5
        return compute_prox_optimized(x, t, f, delta=delta, num_samples=num_samples, 
                                         alpha=alpha, linesearch_iters=linesearch_iters, 
                                         device=device)
    
    w = torch.softmax(z, dim=0)
    
    # More efficient overflow check
    if torch.isinf(w).any():
        alpha *= 0.5
        return compute_prox_optimized(x, t, f, delta=delta, num_samples=num_samples, 
                                         alpha=alpha, linesearch_iters=linesearch_iters, 
                                         device=device)
    
    # Compute proximal term - use einsum for efficiency
    prox_term = torch.einsum('i,ij->j', w, y).unsqueeze(1)
    
    # The experiments use only the proximal estimate.
    return prox_term, linesearch_iters


# ============================================================================
# Helper Functions for Davis-Yin
# ============================================================================

def overlapping_group_penalty_compiled(beta_samples, groups, lambda_2, weights):
    """
    Compute overlapping group penalty with JIT optimization.
    """
    @torch.compile(mode="reduce-overhead")
    def _compute_group_norm(group_samples):
        return torch.linalg.vector_norm(group_samples, dim=0)
    
    if beta_samples.dim() == 1:
        penalty = 0.0
        for i, Gi in enumerate(groups):
            penalty += lambda_2 * weights[i] * _compute_group_norm(beta_samples[Gi].unsqueeze(1)).item()
        return torch.as_tensor(penalty, device=beta_samples.device, dtype=beta_samples.dtype)
    else:
        n_samples = beta_samples.shape[1]
        penalties = torch.zeros(n_samples, device=beta_samples.device, dtype=beta_samples.dtype)
        
        for i, Gi in enumerate(groups):
            group_norms = _compute_group_norm(beta_samples[Gi, :])
            penalties.add_(group_norms, alpha=lambda_2 * weights[i])
        
        return penalties


# ============================================================================
# Algorithm 1: Davis-Yin with HJ-Prox (Adaptive)
# ============================================================================

def davis_yin_adaptive_overlapping_group_lasso(
        X, y, groups, lambda_1, lambda_2,
        weights=None,
        gamma_init=None,
        tau=0.95,
        growth_factor=1.25,
        delta=0.1,
        num_samples=100,
        max_iter=500,
        tol=1e-6,
        z_init=None,
        u_init=None,
        device='cpu',
        verbose=True):
    """
    Adaptive Three Operator Splitting (Algorithm 1, Variant 2).
    
    Problem: minimize f(β) + g(β) + h(β)
    - f(β) = 0.5||Xβ - y||²      (smooth, Lipschitz gradient)
    - g(β) = λ1||β||₁            (proximal, CHEAP - soft threshold)
    - h(β) = λ2 Σⁱ wⁱ||β_Gⁱ||₂   (proximal, EXPENSIVE - group lasso)
    
    Returns:
        dict with keys:
            'beta_z': z_new (recommended solution for structured penalties)
            'beta_x': x (intermediate iterate)
            'dual': u_new (dual variable)
            'history': tracking dict with objective_x, objective_z, gamma, etc.
    """
    
    # Setup
    X_np = np.asarray(X, dtype=np.float64)
    y_np = np.asarray(y, dtype=np.float64).ravel()
    n, p = X_np.shape
    
    X_torch = torch.tensor(X_np, dtype=torch.float64, device=device)
    y_torch = torch.tensor(y_np, dtype=torch.float64, device=device)
    
    if weights is None:
        weights_np = np.array([np.sqrt(len(g)) for g in groups], dtype=np.float64)
    else:
        weights_np = np.asarray(weights, dtype=np.float64)
    
    # Initial step size
    if gamma_init is None:
        L_approx = torch.linalg.norm(X_torch, ord=2)**2
        gamma = 0.9 / L_approx.item()
    else:
        gamma = gamma_init
    
    # Initialize z, u
    if z_init is not None:
        z = torch.tensor(z_init, dtype=torch.float64, device=device)
    else:
        z = torch.zeros(p, 1, device=device, dtype=torch.float64)
    
    if u_init is not None:
        u = torch.tensor(u_init, dtype=torch.float64, device=device)
    else:
        u = torch.zeros(p, 1, device=device, dtype=torch.float64)
    
    if z.dim() == 1:
        z = z.unsqueeze(1)
    if u.dim() == 1:
        u = u.unsqueeze(1)
    
    XTy = X_torch.T @ y_torch.unsqueeze(1)
    
    # Estimate Lipschitz constant of h for Variant 2 growth
    beta_h = lambda_2 * np.sqrt(len(groups))
    
    # Operators
    def grad_f(beta):
        """∇f(β) = X^T(Xβ - y)"""
        b = beta if beta.dim() == 2 else beta.unsqueeze(1)
        return X_torch.T @ (X_torch @ b) - XTy
    
    def eval_f(beta):
        """f(β) = 0.5||Xβ - y||²"""
        b = beta if beta.dim() == 2 else beta.unsqueeze(1)
        resid = X_torch @ b - y_torch.unsqueeze(1)
        return 0.5 * torch.sum(resid ** 2)
    
    def eval_g(beta):
        """g(β) = λ1||β||₁"""
        b = beta if beta.dim() == 2 else beta.unsqueeze(1)
        return lambda_1 * torch.sum(torch.abs(b))
    
    def prox_g(v, step):
        """prox_{γg}(v) for g(β) = λ1||β||₁ - CHEAP (soft-thresholding)"""
        threshold = step * lambda_1
        return torch.sign(v) * torch.maximum(torch.abs(v) - threshold, 
                                              torch.zeros_like(v))
    
    def prox_h(v, step, delta_k):
        """prox_{γh}(v) via HJ-Prox - EXPENSIVE but called once per iteration"""
        v_col = v if v.dim() == 2 else v.unsqueeze(1)
        
        def penalty_func(beta_samples):
            if beta_samples.dim() == 2:
                beta_samples = beta_samples.T
            return overlapping_group_penalty_compiled(
                beta_samples, groups, lambda_2, weights_np
            )
        
        beta_prox, _ = compute_prox_optimized(
            x=v_col, t=step, f=penalty_func,
            delta=delta_k, num_samples=num_samples, device=device
        )
        return beta_prox
    
    # Tracking
    history = {
        'objective_x': [],
        'objective_z': [],
        'gamma': [],
        'backtracks': [],
        'time_prox_h': [],
        'delta_t': []
    }
    
    gamma_prev = gamma
    delta_prev = 0.0
    
    if verbose:
        print("Adaptive Three Operator Splitting (Paper Algorithm 1, Variant 2)")
        print(f"Initial γ: {gamma:.6e}")
        print(f"τ (backtrack): {tau}, β_h (Lipschitz): {beta_h:.2f}")
        print("=" * 80 + "\n")
    
    # Main Loop
    for k in range(max_iter):
        delta_k = delta / (k + 1)**2
        
        # STEP 1: Backtracking Line Search
        if k > 0 and delta_prev > 0:
            gamma_trial = gamma_prev * growth_factor
        else:
            gamma_trial = gamma_prev
        
        backtrack_count = 0
        
        f_z = eval_f(z)
        grad_f_z = grad_f(z)
        
        while True:
            v = z - gamma_trial * u - gamma_trial * grad_f_z
            x = prox_g(v, gamma_trial)
            
            f_x = eval_f(x)
            
            diff = x - z
            linear_term = torch.sum(grad_f_z * diff)
            quadratic_term = (1.0 / (2.0 * gamma_trial)) * torch.sum(diff ** 2)
            Q_t = f_z + linear_term + quadratic_term
            
            if f_x <= Q_t + 1e-12:
                gamma = gamma_trial
                delta_t = (Q_t - f_x).item()
                break
            else:
                gamma_trial = tau * gamma_trial
                backtrack_count += 1
                
                if gamma_trial < 1e-15:
                    if verbose:
                        print(f"Warning: γ too small ({gamma_trial:.2e}), accepting anyway")
                    gamma = gamma_trial
                    delta_t = (Q_t - f_x).item()
                    break
        
        # STEP 2: Apply Expensive Operator
        t0_h = time.time()
        v_h = x + gamma * u
        z_new = prox_h(v_h, gamma, delta_k)
        t_h = time.time() - t0_h
        
        # STEP 3: Update Dual Variable
        u_new = u + (x - z_new) / gamma
        
        # Convergence Check
        with torch.no_grad():
            f_x_val = eval_f(x)
            g_x_val = eval_g(x)
            h_x_val = overlapping_group_penalty_compiled(x, groups, lambda_2, weights_np).sum()
            obj_x = (f_x_val + g_x_val + h_x_val).item()
            
            f_z_val = eval_f(z_new)
            g_z_val = eval_g(z_new)
            h_z_val = overlapping_group_penalty_compiled(z_new, groups, lambda_2, weights_np).sum()
            obj_z = (f_z_val + g_z_val + h_z_val).item()
        
        z_change = torch.norm(z_new - z).item()
        u_change = torch.norm(u_new - u).item()
        
        history['objective_x'].append(obj_x)
        history['objective_z'].append(obj_z)
        history['gamma'].append(gamma)
        history['backtracks'].append(backtrack_count)
        history['time_prox_h'].append(t_h * 1000)
        history['delta_t'].append(delta_t)
        
        if verbose and (k % 10 == 0 or k < 5):
            print(f"Iter {k:3d} | Obj(x): {obj_x:.6f} | Obj(z): {obj_z:.6f} | "
                  f"γ: {gamma:.6e} (bt: {backtrack_count}) | "
                  f"|Δz|: {z_change:.2e} |Δu|: {u_change:.2e} | "
                  f"prox_h: {t_h*1000:.1f}ms")
        
        if z_change < tol and u_change < tol and k > 10:
            if verbose:
                print(f"\n{'='*80}")
                print(f"Converged at iteration {k}")
                print(f"Final objective (z): {obj_z:.6e}")
                print(f"Final objective (x): {obj_x:.6e}")
            break
        
        z = z_new
        u = u_new
        gamma_prev = gamma
        delta_prev = delta_t
    
    return z_new.squeeze().cpu().numpy(), history


# ============================================================================
# Helper Functions for FoGLasso
# ============================================================================

def soft_threshold(v, tau):
    """Soft-thresholding S_tau(v) applied elementwise."""
    return np.sign(v) * np.maximum(np.abs(v) - tau, 0.0)


def estimate_L_xtx(X, iters=25, seed=0):
    """
    Power iteration estimate of ||X^T X||_2 = sigma_max(X)^2 (no SVD needed).
    """
    rng = np.random.default_rng(seed)
    p = X.shape[1]
    v = rng.normal(size=p)
    v /= np.linalg.norm(v) + 1e-12
    for _ in range(iters):
        v = X.T @ (X @ v)
        v /= (np.linalg.norm(v) + 1e-12)
    XtXv = X.T @ (X @ v)
    return float(v @ XtXv)


def preprocess_zero_groups(u, groups, lambda_2, weights, max_iter=100, tol=0.0):
    """
    Paper-faithful screening (Lemma 3 iterative procedure):
    Cycle through groups; if ||u_Gi|| <= lambda_2 * w_i, set u_Gi = 0.
    Repeat until u does not change (or max_iter reached).
    """
    u = u.copy()
    g = len(groups)
    group_zero_mask = np.zeros(g, dtype=bool)

    changed = True
    it = 0
    while changed and it < max_iter:
        changed = False
        for i, Gi in enumerate(groups):
            if group_zero_mask[i]:
                continue
            if np.linalg.norm(u[Gi]) <= (lambda_2 * weights[i] + tol):
                group_zero_mask[i] = True
                u[Gi] = 0.0
                changed = True
        it += 1

    active_indices = np.flatnonzero(u)
    if active_indices.size == 0:
        return u[active_indices], [], np.array([], dtype=int), np.array([], dtype=float)

    index_map = -np.ones(u.shape[0], dtype=np.int64)
    index_map[active_indices] = np.arange(active_indices.size)

    active_groups = []
    active_weights = []
    for i, Gi in enumerate(groups):
        if group_zero_mask[i]:
            continue
        Gi_new = index_map[Gi]
        Gi_new = Gi_new[Gi_new >= 0]
        if Gi_new.size == 0:
            continue
        active_groups.append(Gi_new)
        active_weights.append(weights[i])

    return u[active_indices], active_groups, active_indices, np.asarray(active_weights)


def _dual_lipschitz_overlap_bound(p, groups):
    """
    A safe Lipschitz bound for the dual gradient based on maximum overlap count:
    L_dual <= max_j #{i : j in G_i}
    """
    if p <= 0 or len(groups) == 0:
        return 0.0
    counts = np.zeros(p, dtype=np.int32)
    for Gi in groups:
        counts[Gi] += 1
    return float(counts.max()) if counts.size else 0.0


def solve_dual_problem_sparse(u, groups, lambda_2, weights,
                              max_iter=2000,
                              gap_tol=1e-10,
                              check_every=1,
                              verbose=False):
    """
    Solve the smooth dual (Eq. (15)) with accelerated projected gradient (AGD),
    and recover primal x via x = max(u - Y e, 0) (Eq. (14)).

    Terminate when the estimated duality gap < gap_tol (Theorem 2).
    """
    u = np.asarray(u)
    p = len(u)
    g = len(groups)
    dtype = u.dtype

    if p == 0 or g == 0:
        return u.copy(), [np.zeros(0, dtype=dtype) for _ in range(g)], {'duality_gap': []}

    L_dual = _dual_lipschitz_overlap_bound(p, groups)
    if L_dual <= 0.0:
        return u.copy(), [np.zeros(0, dtype=dtype) for _ in range(g)], {'duality_gap': []}
    step = 1.0 / L_dual

    # FISTA/AGD sequences on Y
    Y_current = [np.zeros(len(Gi), dtype=dtype) for Gi in groups]
    Y_momentum = [y.copy() for y in Y_current]
    t = 1.0

    hist = {'duality_gap': []}

    for it in range(max_iter):
        # Y_sum = Y e (accumulate overlaps)
        Y_sum = np.zeros(p, dtype=dtype)
        for i, Gi in enumerate(groups):
            Y_sum[Gi] += Y_momentum[i]

        # Primal from current dual iterate
        x = np.maximum(u - Y_sum, 0.0)

        # Gradient step on dual + projection onto Omega
        Y_new = []
        Y_sum_new = np.zeros(p, dtype=dtype)
        for i, Gi in enumerate(groups):
            y = Y_momentum[i] + step * x[Gi]
            r = lambda_2 * weights[i]
            nrm = np.linalg.norm(y)
            if nrm > r:
                y = (r / nrm) * y
            Y_new.append(y)
            Y_sum_new[Gi] += y

        # Duality gap check
        if (it % check_every) == 0:
            x_new = np.maximum(u - Y_sum_new, 0.0)
            gap = 0.0
            for i, Gi in enumerate(groups):
                xi = x_new[Gi]
                gap += (lambda_2 * weights[i]) * np.linalg.norm(xi) - float(np.dot(xi, Y_new[i]))

            hist['duality_gap'].append(gap)

            if verbose:
                print(f"[prox-dual] it={it} gap={gap:.3e}")

            if gap < gap_tol:
                Y_current = Y_new
                break

        # FISTA extrapolation
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t * t)) / 2.0
        beta = (t - 1.0) / t_new
        Y_momentum = [Y_new[i] + beta * (Y_new[i] - Y_current[i]) for i in range(g)]
        Y_current = Y_new
        t = t_new

    # Recover primal solution
    Y_sum = np.zeros(p, dtype=dtype)
    for i, Gi in enumerate(groups):
        Y_sum[Gi] += Y_current[i]
    x_prox = np.maximum(u - Y_sum, 0.0)

    return x_prox, Y_current, hist


def compute_proximal_operator_foglasso(v, groups, lambda_1, lambda_2, weights,
                                       prox_max_iter=2000,
                                       prox_gap_tol=1e-5,
                                       screen_max_iter=100,
                                       verbose=False):
    """
    Proximal operator π_{λ1,λ2}(v) with screening and dual solving.

    Implements:
      1) Theorem 1: reduce λ1>0 via soft-thresholding and handle sign
      2) Lemma 3: iterative screening of zero groups
      3) Solve smooth dual (Eq. (15)) via AGD and recover primal (Eq. (14))
      4) Restore sign
    """
    v = np.asarray(v)
    groups = [np.asarray(g, dtype=np.int64) for g in groups]
    weights = np.asarray(weights, dtype=float)

    if lambda_1 < 0 or lambda_2 < 0:
        raise ValueError("lambda_1 and lambda_2 must be >= 0.")
    if len(groups) != len(weights):
        raise ValueError("weights must have the same length as groups.")

    # Theorem 1 reduction + sign handling
    sgn = np.sign(v)
    u = np.maximum(np.abs(v) - lambda_1, 0.0)

    if not np.any(u):
        return np.zeros_like(v), {'duality_gap': []}

    if lambda_2 == 0.0 or len(groups) == 0:
        return sgn * u, {'duality_gap': []}

    # Lemma 3 screening
    u_red, active_groups, active_indices, active_w = preprocess_zero_groups(
        u, groups, lambda_2, weights, max_iter=screen_max_iter, tol=0.0
    )

    if active_indices.size == 0:
        return np.zeros_like(v), {'duality_gap': []}

    # Dual solve on reduced problem
    x_red, _, hist = solve_dual_problem_sparse(
        u_red, active_groups, lambda_2, active_w,
        max_iter=prox_max_iter, gap_tol=prox_gap_tol, check_every=1, verbose=verbose
    )

    # Reconstruct full solution and restore sign
    x_abs = np.zeros_like(v, dtype=u.dtype)
    x_abs[active_indices] = x_red

    return sgn * x_abs, hist


# ============================================================================
# Algorithm 2: FoGLasso (Dual Solver with Analytical Proximal Operators) (Implemented exactly as in the 2011 NeurIPS paper)
# ============================================================================

def foglasso(X, y, groups, lambda_1, lambda_2, weights=None,
             L_init=1.0,
             max_iter=1000,
             obj_tol=1e-5,
             prox_gap_tol=1e-5,
             prox_max_iter=1000,
             verbose=True):
    """
    Paper-faithful FoGLasso for least squares:

        min_x  (1/2)||Xx - y||^2 + λ1||x||_1 + λ2 Σ_i w_i ||x_{G_i}||_2

    Uses backtracking line search and dual solver for proximal operator.
    Terminates when adjacent objective change <= obj_tol.
    """
    X = np.asarray(X)
    y = np.asarray(y)
    n, p = X.shape

    groups = [np.asarray(g, dtype=np.int64) for g in groups]
    if weights is None:
        weights = np.array([np.sqrt(len(g)) for g in groups], dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    # Initialize
    x_prev = np.zeros(p, dtype=float)
    x = x_prev.copy()

    alpha_old = 0.0
    alpha = 1.0

    L_prev = float(L_init)
    if L_prev <= 0:
        raise ValueError("L_init must be > 0.")

    hist = {
        "objective": [],
        "loss": [],
        "penalty": [],
        "L": [],
    }

    def penalty_val(z):
        val = lambda_1 * np.sum(np.abs(z))
        for i, Gi in enumerate(groups):
            val += lambda_2 * weights[i] * np.linalg.norm(z[Gi])
        return float(val)

    def loss_and_grad(z):
        r = X @ z - y
        loss = 0.5 * float(np.dot(r, r))
        grad = X.T @ r
        return loss, grad

    # Record initial objective
    loss0, _ = loss_and_grad(x)
    obj0 = loss0 + penalty_val(x)
    hist["objective"].append(obj0)
    hist["loss"].append(loss0)
    hist["penalty"].append(obj0 - loss0)
    hist["L"].append(L_prev)

    for it in range(1, max_iter + 1):
        # Step 3: s_i = x_i + β_i (x_i - x_{i-1})
        beta = (alpha_old - 1.0) / alpha
        s = x + beta * (x - x_prev)

        # Gradient at s_i
        loss_s, grad_s = loss_and_grad(s)

        # Step 4: backtracking line search
        L_trial = L_prev

        while True:
            v = s - (1.0 / L_trial) * grad_s

            x_new, prox_hist = compute_proximal_operator_foglasso(
                v,
                groups,
                lambda_1=lambda_1 / L_trial,
                lambda_2=lambda_2 / L_trial,
                weights=weights,
                prox_max_iter=prox_max_iter,
                prox_gap_tol=prox_gap_tol,
                verbose=False
            )

            # Check majorization
            loss_new, _ = loss_and_grad(x_new)
            dx = x_new - s
            rhs = loss_s + float(np.dot(grad_s, dx)) + 0.5 * L_trial * float(np.dot(dx, dx))

            if loss_new <= rhs:
                break

            L_trial *= 2.0

        L_prev = L_trial

        # Step 5: α_{i+1} = (1 + sqrt(1 + 4 α_i^2))/2
        alpha_new = (1.0 + np.sqrt(1.0 + 4.0 * alpha * alpha)) / 2.0

        # Objective bookkeeping
        pen = penalty_val(x_new)
        obj = loss_new + pen

        hist["objective"].append(obj)
        hist["loss"].append(loss_new)
        hist["penalty"].append(pen)
        hist["L"].append(L_prev)

        if verbose and (it % 10 == 0 or it == 1):
            obj_change = abs(hist["objective"][-1] - hist["objective"][-2])
            print(f"Iter {it:4d}: Obj={obj:.6f}, Loss={loss_new:.6e}, "
                  f"Pen={pen:.6f}, |ΔObj|={obj_change:.2e}, L={L_prev:.2e}")

        # Stopping criterion
        if abs(hist["objective"][-1] - hist["objective"][-2]) <= obj_tol:
            if verbose:
                print(f"Converged at iter {it} (|ΔObj| <= {obj_tol:g}).")
            x = x_new
            break

        x_prev = x
        x = x_new
        alpha_old = alpha
        alpha = alpha_new

    return x, hist


print("✓ All algorithms and helper functions loaded successfully")

## Data loading (GSE2034 + KEGG pathways)


In [ ]:
# ============================================================================
# CHUNK 2: DATA LOADING
# ============================================================================

def get_pathway_groups(gene_names):
    """Get KEGG pathway groups for genes."""
    pathways = gp.get_library(name='KEGG_2021_Human')
    gene_to_idx = {gene: i for i, gene in enumerate(gene_names)}

    groups = []
    pathway_names = []

    for pathway_name, pathway_genes in pathways.items():
        idx = [gene_to_idx[g] for g in pathway_genes if g in gene_to_idx]
        idx = np.unique(idx)
        idx = np.sort(idx).astype(int)

        if 5 <= idx.size <= 200:
            groups.append(idx)
            pathway_names.append(pathway_name)

    return groups, pathway_names


def load_real_gse2034_data_FIXED():
    """Load GSE2034 with correct relapse labels."""
    print("Downloading GSE2034 (Breast Cancer)...")
    gse = GEOparse.get_GEO(geo="GSE2034", destdir="./data")
    
    print("Extracting clinical labels from metadata...")
    y_labels = []
    sample_names = []
    
    for gsm_name, gsm in gse.gsms.items():
        chars = gsm.metadata.get('characteristics_ch1', [])
        
        relapse_status = None
        for char in chars:
            char_lower = str(char).lower()
            if 'relapse' in char_lower and ':' in char_lower:
                value = char_lower.split(':')[-1].strip()
                if value in ['0', '1']:
                    relapse_status = int(value)
                    break
        
        if relapse_status is not None:
            y_labels.append(relapse_status)
            sample_names.append(gsm_name)
    
    y = np.array(y_labels)
    print(f"Extracted labels for {len(y)} samples")
    print(f"Class distribution: {np.bincount(y)} (0=No Relapse, 1=Relapse)")
    
    if len(y) == 0 or len(np.unique(y)) < 2:
        raise ValueError(f"Label extraction failed! Got {len(y)} samples with {len(np.unique(y))} classes")
    
    # Build expression matrix
    print("Building expression matrix...")
    pivoted_data = gse.pivot_samples('VALUE')
    pivoted_data = pivoted_data[sample_names]
    
    # Map probes to genes
    print("Mapping probes to gene symbols...")
    platform_name = list(gse.gpls.keys())[0]
    gpl = gse.gpls[platform_name]
    
    probe_to_gene = {}
    for _, row in gpl.table.iterrows():
        sym = None
        for col in ['Gene Symbol', 'GENE_SYMBOL', 'Symbol']:
            if col in row and pd.notna(row[col]) and row[col] != '':
                sym = str(row[col]).split('///')[0].strip()
                break
        if sym and sym != '---':
            probe_to_gene[row['ID']] = sym
    
    pivoted_data['Gene'] = pivoted_data.index.map(probe_to_gene)
    pivoted_data = pivoted_data.dropna(subset=['Gene'])
    
    # Average duplicates
    gene_expression_df = pivoted_data.groupby('Gene').mean()
    
    X = gene_expression_df.T.values
    gene_names = gene_expression_df.index.tolist()
    
    # Log transform if needed
    if np.max(X) > 100:
        print("Applying Log2(x+1) transform...")
        X = np.log2(X + 1)
    
    X = StandardScaler().fit_transform(X)
    
    print(f"Final: {X.shape[0]} samples x {X.shape[1]} genes")
    print(f"Verification: y has {np.sum(y==0)} non-relapse, {np.sum(y==1)} relapse")
    
    return X, y, gene_names


def load_gse2034_filtered(top_k=1000, method='ttest'):
    """Load GSE2034 with feature selection."""
    X, y, gene_names = load_real_gse2034_data_FIXED()
    
    if method == 'variance':
        variances = np.var(X, axis=0)
        top_indices = np.argsort(variances)[-top_k:]
    elif method == 'ttest':
        from scipy.stats import ttest_ind
        t_stats = []
        for i in range(X.shape[1]):
            t_stat, _ = ttest_ind(X[y==0, i], X[y==1, i])
            t_stats.append(abs(t_stat))
        top_indices = np.argsort(t_stats)[-top_k:]
    else:
        raise ValueError(f"Unknown method: {method}")
    
    X_filtered = X[:, top_indices]
    gene_names_filtered = [gene_names[i] for i in top_indices]
    
    print(f"Reduced using {method}: {X.shape[1]} → {X_filtered.shape[1]} genes")
    return X_filtered, y, gene_names_filtered


print("\n" + "="*80)
print("Loading GSE2034 Breast Cancer Data...")
print("="*80)

X, y, gene_names = load_gse2034_filtered(13237)
groups, pathway_names = get_pathway_groups(gene_names)

# Transform Y for numerical stability
y = y * 2 - 1

print(f"\n✓ Data loaded: {X.shape[0]} samples, {X.shape[1]} genes")
print(f"✓ Pathways: {len(groups)} overlapping groups")
print(f"✓ Response transformed: y ∈ {{-1, +1}}")

# Set parameters
lambda_1 = 0.005
lambda_2 = 0.0001

print(f"✓ Penalty parameters: λ1={lambda_1}, λ2={lambda_2}")

## Algorithm 1 — FoGLasso (analytical dual solver)


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - FoGLasso
# ============================================================================

print("\n" + "="*80)
print("Running Algorithm 1: FoGLasso (Dual Solver)...")
print("="*80)

start_time = time.time()

beta_FoGLasso, hist_FoGLasso = foglasso(
    X, y, groups, lambda_1=lambda_1, lambda_2=lambda_2, L_init=1.0,
    max_iter=10000, verbose=True, obj_tol=1e-5
)

elapsed_time = time.time() - start_time

print(f"\n✓ FoGLasso completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(hist_FoGLasso['objective'])-1} iterations")
print(f"  - Final objective: {hist_FoGLasso['objective'][-1]:.6f}")
print(f"  - Final penalty: {hist_FoGLasso['penalty'][-1]:.6f}")
print(f"  - Final loss: {hist_FoGLasso['loss'][-1]:.6f}")
print(f"  - ||β||: {np.linalg.norm(beta_FoGLasso):.4f}")
print(f"  - Non-zero coefficients: {np.count_nonzero(beta_FoGLasso)}")

## Algorithm 2 — DYS with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - Davis-Yin with HJ-Prox
# ============================================================================

print("\n" + "="*80)
print("Running Algorithm 2: Davis-Yin with HJ-Prox...")
print("="*80)

start_time = time.time()

beta_HJ, hist_HJ = davis_yin_adaptive_overlapping_group_lasso(
    X, y, groups,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    delta=10,
    num_samples=1000,
    max_iter=100000,
    verbose=True,
    tol=1e-1000
)

elapsed_time = time.time() - start_time

print(f"\n✓ Davis-Yin HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(hist_HJ['objective_z'])} iterations")
print(f"  - Final objective: {hist_HJ['objective_z'][-1]:.6f}")

## Pathway analysis


In [ ]:
# ============================================================================
# CHUNK 5: ANALYSIS - Active Pathway Identification
# ============================================================================

def analyze_active_pathways(beta_fog, groups, pathway_names, gene_names, top_n=20, active_tol=1e-6):
    """
    Identify and analyze the most active pathways in results.
    
    Parameters:
    -----------
    beta_fog : array - Solution coefficients
    groups : list of arrays - group indices for each pathway
    pathway_names : list of str - pathway names corresponding to groups
    gene_names : list of str - gene symbols
    top_n : int - number of top pathways to display
    active_tol : float - tolerance for considering a group active
    
    Returns:
    --------
    results_df : DataFrame with pathway analysis
    """
    print("="*80)
    print("ANALYZING ACTIVE PATHWAYS")
    print("="*80)
    
    # Compute group norms
    group_norms = np.array([np.linalg.norm(beta_fog[np.asarray(Gi, dtype=int)]) for Gi in groups])
    
    # Compute pathway sizes
    pathway_sizes = np.array([len(Gi) for Gi in groups])
    
    # Compute normalized group norms (SIZE-ADJUSTED)
    normalized_group_norms = group_norms / np.sqrt(pathway_sizes)
    
    # Overall statistics
    n_active = np.sum(group_norms > active_tol)
    print(f"\nOverall Statistics:")
    print(f"  Total pathways: {len(groups)}")
    print(f"  Active pathways: {n_active} ({100*n_active/len(groups):.1f}%)")
    print(f"  Inactive pathways: {len(groups) - n_active}")
    
    print(f"\nGroup norm distribution:")
    print(f"  Min: {group_norms.min():.3e}")
    print(f"  Q1: {np.percentile(group_norms, 25):.3e}")
    print(f"  Median: {np.median(group_norms):.3e}")
    print(f"  Q3: {np.percentile(group_norms, 75):.3e}")
    print(f"  Max: {group_norms.max():.3e}")
    
    # Build results table
    results = []
    for i, Gi in enumerate(groups):
        pathway_name = pathway_names[i]
        group_norm = group_norms[i]
        normalized_norm = normalized_group_norms[i]
        pathway_size = pathway_sizes[i]
        
        genes_in_pathway = [gene_names[idx] for idx in Gi]
        coeffs_in_pathway = beta_fog[Gi]
        
        n_nonzero = np.sum(np.abs(coeffs_in_pathway) > 1e-10)
        avg_abs_coeff = np.mean(np.abs(coeffs_in_pathway))
        max_abs_coeff = np.max(np.abs(coeffs_in_pathway))
        
        results.append({
            'pathway_index': i,
            'pathway_name': pathway_name,
            'group_norm': group_norm,
            'normalized_group_norm': normalized_norm,
            'pathway_size': pathway_size,
            'n_nonzero_genes': n_nonzero,
            'pct_genes_active': 100 * n_nonzero / pathway_size,
            'avg_abs_coeff': avg_abs_coeff,
            'max_abs_coeff': max_abs_coeff,
            'genes': genes_in_pathway,
            'coefficients': coeffs_in_pathway
        })
    
    results_df = pd.DataFrame(results)
    
    # Display rankings
    print(f"\n{'='*80}")
    print(f"TOP {top_n} PATHWAYS BY RAW GROUP NORM (||β_G||)")
    print(f"{'='*80}")
    print("⚠️  WARNING: This ranking is BIASED by pathway size!")
    print("-"*80)
    
    results_by_raw = results_df.sort_values('group_norm', ascending=False)
    print(f"\n{'Rank':<6} {'Pathway':<60} {'||β_G||':<12}")
    print("-"*80)
    for rank, (idx, row) in enumerate(results_by_raw.head(top_n).iterrows(), 1):
        pathway_short = row['pathway_name'][:58] + ".." if len(row['pathway_name']) > 60 else row['pathway_name']
        print(f"{rank:<6} {pathway_short:<60} {row['group_norm']:<12.4e}")
    
    print(f"\n{'='*80}")
    print(f"TOP {top_n} PATHWAYS BY NORMALIZED GROUP NORM (||β_G|| / √|G|)")
    print(f"{'='*80}")
    print("✅ RECOMMENDED: This ranking is SIZE-ADJUSTED and biologically meaningful!")
    print("-"*80)
    
    results_by_normalized = results_df.sort_values('normalized_group_norm', ascending=False)
    print(f"\n{'Rank':<6} {'Pathway':<60} {'Norm/√Size':<12}")
    print("-"*80)
    for rank, (idx, row) in enumerate(results_by_normalized.head(top_n).iterrows(), 1):
        pathway_short = row['pathway_name'][:58] + ".." if len(row['pathway_name']) > 60 else row['pathway_name']
        print(f"{rank:<6} {pathway_short:<60} {row['normalized_group_norm']:<12.4e}")
    
    return results_by_normalized


print("\n" + "="*80)
print("Analyzing FoGLasso Results...")
print("="*80)

results_FoGLasso = analyze_active_pathways(
    beta_fog=beta_FoGLasso,
    groups=groups,
    pathway_names=pathway_names,
    gene_names=gene_names,
    top_n=3
)

print("\n" + "="*80)
print("Analyzing Davis-Yin HJ Results...")
print("="*80)

results_HJ = analyze_active_pathways(
    beta_fog=beta_HJ,
    groups=groups,
    pathway_names=pathway_names,
    gene_names=gene_names,
    top_n=3
)


## Comparison plots


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 6: GENERATE FIGURES
# ============================================================================

print("\n" + "="*80)
print("Generating figures...")
print("="*80)


def correlation_scatter(
    beta1,
    beta2,
    labels=('HJ-Prox', 'FoGLasso'),
    figsize=(8, 8),
    diagonal_color='#B71C1C',
    diagonal_style='--',
    diagonal_width=2.5,
    point_size=20,
    point_alpha=0.5,
    show_legend=True,
    fontsize_axes=14,
    fontsize_title=16,
    fontsize_corr=12
):
    """
    Draw coefficient correlation scatter plot.

    Returns:
        fig: matplotlib.figure.Figure
    """
    b1 = np.asarray(beta1).ravel()
    b2 = np.asarray(beta2).ravel()
    assert b1.shape == b2.shape, "beta1 and beta2 must have same shape"

    pearson_corr = np.corrcoef(b1, b2)[0, 1]

    fig, ax = plt.subplots(figsize=figsize)

    ax.scatter(b1, b2, s=point_size, alpha=point_alpha, edgecolors='none')

    # Diagonal (perfect agreement)
    lim_min = min(b1.min(), b2.min())
    lim_max = max(b1.max(), b2.max())
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        linestyle=diagonal_style,
        linewidth=diagonal_width,
        color=diagonal_color,
        alpha=0.9,
        label='Perfect agreement' if show_legend else None
    )

    ax.set_xlabel(f'{labels[0]} coefficients', fontsize=fontsize_axes, fontweight='bold')
    ax.set_ylabel(f'{labels[1]} coefficients', fontsize=fontsize_axes, fontweight='bold')
    ax.set_title('Coefficient Comparison', fontsize=fontsize_title, fontweight='bold')

    # Pearson correlation text
    ax.text(
        0.02, 0.98,
        f'Pearson r = {pearson_corr:.5f}',
        transform=ax.transAxes,
        ha='left', va='top',
        fontsize=fontsize_corr, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', alpha=0.15)
    )

    ax.axhline(0, color='k', lw=0.8, alpha=0.3)
    ax.axvline(0, color='k', lw=0.8, alpha=0.3)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')

    if show_legend:
        ax.legend()

    fig.tight_layout()
    return fig


# --- Figure 1: Coefficient Correlation ---
fig = correlation_scatter(beta_HJ, beta_FoGLasso,
                          labels=('Davis-Yin (HJ-Prox)', 'FoGLasso (Dual)'),
                          figsize=(8, 8))
fig.savefig('figures/coeff_correlation.pdf', bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: Objective Convergence ---
limit = 1000
obj_FoGLasso = hist_FoGLasso["objective"][:limit]
obj_HJ = hist_HJ["objective_z"][:limit]

plt.figure(figsize=(10, 8))

plt.semilogy(obj_FoGLasso, '-', linewidth=3,
             label=f'FOGLASSO: {obj_FoGLasso[-1]:.3f}')
plt.semilogy(obj_HJ, '--', linewidth=3,
             label=f'DYS-HJ: {obj_HJ[-1]:.3f}')

plt.ylabel('Objective Value (log scale)', fontsize=30)
plt.xlabel('Iteration', fontsize=40)
plt.title('Overlapping Group LASSO', fontsize=40)
plt.legend(fontsize=35, loc='upper right')
plt.grid(True, alpha=0.3, which='both')
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/OGLASSO_objective_convergence.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: coeff_correlation.pdf, OGLASSO_objective_convergence.pdf")
print("\n" + "="*80)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*80)